In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_4.pickle

In [ ]:
%%cudf.pandas.profile
### cell 4 ###

def create_dataframe_of_counts_16(dataframe, column, rename_index, rename_column, return_percentages=False):
    # compute counts (or normalized counts) entirely on GPU
    vc = dataframe[column].value_counts(normalize=return_percentages)
    if return_percentages:
        # convert proportions to percentages on GPU
        vc = vc * 100
    # reset_index already returns a GPU DataFrame
    df_counts = vc.reset_index()
    # rename columns on GPU
    return df_counts.rename(columns={
        'index': rename_index,
        column:    rename_column
    })

# call the optimized function
percentages_per_country_df = create_dataframe_of_counts_16(
    responses_df_2022_cell10[1:],
    'In which country do you currently reside?',
    'country',
    '% of respondents',
    return_percentages=False
)

# inspect the resulting GPU DataFrame
percentages_per_country_df.info()